# Exp 1 Reproducibility Check

This notebook reruns `code/01_experiment1_descriptive.R` in an isolated scratch workspace
and compares the rerun outputs against `expected/exp1_descriptive.rds`. Figure checks are
based on figures regenerated from the expected baseline result, not on external `out/` artifacts.

In [ ]:
suppressPackageStartupMessages({
  library(survey)
  library(tidyverse)
  library(patchwork)
})

helper_candidates <- c("repro_utils.R", file.path("notebooks", "repro", "repro_utils.R"))
helper_path <- helper_candidates[file.exists(helper_candidates)][1]
if (is.na(helper_path)) {
  stop("repro_utils.R not found.")
}
source(helper_path)

baseline_result_path <- path_in_repo("expected", "exp1_descriptive.rds")
scratch_dir <- new_scratch("exp1_repro")

cat("Scratch workspace:", scratch_dir, "\n")
cat("Baseline result:", baseline_result_path, "\n")

In [ ]:
run_script_in_scratch(
  "code/01_experiment1_descriptive.R",
  scratch_dir,
  c("data/nhanes_processed.rds", "data/survey_design.rds")
)

baseline_exp1 <- readRDS(baseline_result_path)
rerun_exp1 <- readRDS(file.path(scratch_dir, "results", "exp1_descriptive.rds"))

In [ ]:
numeric_checks <- list(
  compare_df_exact("exp1_continuous", rerun_exp1$continuous, baseline_exp1$continuous),
  compare_df_exact("exp1_prevalence", rerun_exp1$prevalence, baseline_exp1$prevalence),
  compare_df_exact("exp1_age_group", rerun_exp1$age_group, baseline_exp1$age_group),
  compare_df_exact("exp1_race_eth", rerun_exp1$race_eth, baseline_exp1$race_eth)
)

bind_rows(numeric_checks)

In [ ]:
render_exp1_figures <- function(exp1_obj, out_dir) {
  ensure_dir(out_dir)

  plot_data <- exp1_obj$continuous %>%
    select(Variable, Unweighted = Unwt_Mean, Weighted = Wt_Mean) %>%
    pivot_longer(-Variable, names_to = "Type", values_to = "Mean")

  se_data <- exp1_obj$continuous %>%
    select(Variable, Unweighted = Unwt_SE, Weighted = Wt_SE) %>%
    pivot_longer(-Variable, names_to = "Type", values_to = "SE")

  plot_data <- plot_data %>% left_join(se_data, by = c("Variable", "Type"))
  plot_data$Type <- factor(plot_data$Type, levels = c("Unweighted", "Weighted"))

  p1 <- ggplot(plot_data, aes(x = Variable, y = Mean, fill = Type)) +
    geom_col(position = position_dodge(0.8), width = 0.7) +
    geom_errorbar(aes(ymin = Mean - 1.96 * SE, ymax = Mean + 1.96 * SE),
                  position = position_dodge(0.8), width = 0.2) +
    facet_wrap(~Variable, scales = "free_y", nrow = 1) +
    scale_fill_manual(values = c("Unweighted" = "#4DBBD5", "Weighted" = "#E64B35")) +
    labs(y = "Mean", x = NULL, fill = NULL,
         title = "Weighted vs Unweighted Means for Continuous Variables") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom",
          strip.text = element_text(face = "bold"),
          axis.text.x = element_blank(),
          axis.ticks.x = element_blank())

  prev_plot <- exp1_obj$prevalence %>%
    select(Outcome, Unweighted = Unwt_Pct, Weighted = Wt_Pct) %>%
    pivot_longer(-Outcome, names_to = "Type", values_to = "Prevalence")

  prev_se <- exp1_obj$prevalence %>%
    select(Outcome, Unweighted = Unwt_SE, Weighted = Wt_SE) %>%
    pivot_longer(-Outcome, names_to = "Type", values_to = "SE")

  prev_plot <- prev_plot %>% left_join(prev_se, by = c("Outcome", "Type"))
  prev_plot$Type <- factor(prev_plot$Type, levels = c("Unweighted", "Weighted"))

  p2 <- ggplot(prev_plot, aes(x = Outcome, y = Prevalence, fill = Type)) +
    geom_col(position = position_dodge(0.8), width = 0.6) +
    geom_errorbar(aes(ymin = Prevalence - 1.96 * SE, ymax = Prevalence + 1.96 * SE),
                  position = position_dodge(0.8), width = 0.2) +
    scale_fill_manual(values = c("Unweighted" = "#4DBBD5", "Weighted" = "#E64B35")) +
    labs(y = "Prevalence (%)", x = NULL, fill = NULL,
         title = "Weighted vs Unweighted Disease Prevalence") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom")

  age_plot <- exp1_obj$age_group %>%
    select(Group = Age_Group, Sample = Unwt_Pct, Population = Wt_Pct) %>%
    pivot_longer(-Group, names_to = "Type", values_to = "Pct")
  age_plot$Type <- factor(age_plot$Type, levels = c("Sample", "Population"))

  p3a <- ggplot(age_plot, aes(x = Group, y = Pct, fill = Type)) +
    geom_col(position = position_dodge(0.8), width = 0.6) +
    scale_fill_manual(values = c("Sample" = "#4DBBD5", "Population" = "#E64B35")) +
    labs(y = "Proportion (%)", x = NULL, fill = NULL,
         title = "Age Group Composition") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom")

  race_plot <- exp1_obj$race_eth %>%
    select(Group = Race, Sample = Unwt_Pct, Population = Wt_Pct) %>%
    pivot_longer(-Group, names_to = "Type", values_to = "Pct")
  race_plot$Type <- factor(race_plot$Type, levels = c("Sample", "Population"))

  p3b <- ggplot(race_plot, aes(x = Group, y = Pct, fill = Type)) +
    geom_col(position = position_dodge(0.8), width = 0.6) +
    scale_fill_manual(values = c("Sample" = "#4DBBD5", "Population" = "#E64B35")) +
    labs(y = "Proportion (%)", x = NULL, fill = NULL,
         title = "Race/Ethnicity Composition") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom",
          axis.text.x = element_text(angle = 30, hjust = 1))

  p3 <- p3a + p3b + plot_layout(ncol = 2, guides = "collect") &
    theme(legend.position = "bottom")

  paths <- c(
    fig_exp1_continuous = file.path(out_dir, "fig_exp1_continuous.pdf"),
    fig_exp1_prevalence = file.path(out_dir, "fig_exp1_prevalence.pdf"),
    fig_exp1_composition_age = file.path(out_dir, "fig_exp1_composition_age.pdf"),
    fig_exp1_composition_race = file.path(out_dir, "fig_exp1_composition_race.pdf"),
    fig_exp1_composition = file.path(out_dir, "fig_exp1_composition.pdf")
  )

  ggsave(paths[["fig_exp1_continuous"]], p1, width = 10, height = 4)
  ggsave(paths[["fig_exp1_prevalence"]], p2, width = 6, height = 4)
  ggsave(paths[["fig_exp1_composition_age"]], p3a, width = 6, height = 4)
  ggsave(paths[["fig_exp1_composition_race"]], p3b, width = 8, height = 4)
  ggsave(paths[["fig_exp1_composition"]], p3, width = 12, height = 5)

  paths
}

rerun_paths <- render_exp1_figures(rerun_exp1, file.path(scratch_dir, "out"))
baseline_paths <- render_exp1_figures(baseline_exp1, file.path(scratch_dir, "baseline_out"))

figure_checks <- list(
  compare_pdf_figure("fig_exp1_continuous", rerun_paths[["fig_exp1_continuous"]], baseline_paths[["fig_exp1_continuous"]]),
  compare_pdf_figure("fig_exp1_prevalence", rerun_paths[["fig_exp1_prevalence"]], baseline_paths[["fig_exp1_prevalence"]]),
  compare_pdf_figure("fig_exp1_composition_age", rerun_paths[["fig_exp1_composition_age"]], baseline_paths[["fig_exp1_composition_age"]]),
  compare_pdf_figure("fig_exp1_composition_race", rerun_paths[["fig_exp1_composition_race"]], baseline_paths[["fig_exp1_composition_race"]]),
  compare_pdf_figure("fig_exp1_composition", rerun_paths[["fig_exp1_composition"]], baseline_paths[["fig_exp1_composition"]])
)

bind_rows(figure_checks)

In [ ]:
exp1_summary <- summarize_results(c(numeric_checks, figure_checks))
exp1_summary$results